# 07. Machine learning: does a more flexible model do better?

Notebook 05 used a **logistic regression**, which assumes each predictor
shifts the log-odds of disease by a fixed amount. That's easy to interpret, but
it can't capture interactions (a predictor mattering more for some patients
than others) or thresholds (risk jumping above a certain heart rate) unless
you build them in by hand.

**Tree-based models** learn those patterns on their own:

- **Random forest:** hundreds of decision trees, each trained on a random
  resample of patients and a random subset of predictors, then averaged.
- **Gradient boosting:** small trees built one after another, each one
  correcting the mistakes of the ones before it.

**The question:** on this data, does that extra flexibility buy better
predictions? Or does the simple model hold its own?

**How to make the comparison fair:**
1. **Same patients, same splits.** Every model is scored on exactly the same
   20 × 10-fold cross-validation splits, so differences come from the models,
   not from luck in how patients were divided.
2. **Scored on patients the model never saw** (as in notebook 05).
3. **Two measures,** because a good model needs both:
   - **AUC:** does it *rank* patients well?
   - **Brier score:** are its predicted *probabilities* accurate? (The average
     squared gap between predicted risk and what happened; lower is better.)
4. **No tuning on the test data.** Hyperparameters are set to sensible,
   conservative values up front (listed below) rather than tuned, because
   tuning on only 297 patients would itself over-fit. Tuning properly would
   need *nested* cross-validation, noted as a next step.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from clean_data import load_clean  # noqa: E402

df = load_clean()
cle = df[df["site"] == "cleveland"].dropna(subset=["major_vessels", "thallium_result"]).copy()
y = cle["disease"].astype(int).to_numpy()
print(f"{len(cle)} patients, {y.sum()} with disease")

## 1. Predictors and preprocessing

The same two predictor sets as notebook 05: **pre-imaging** (the 11 routine
measurements) and **all 13** (adding vessel count and thallium).

A scikit-learn **`Pipeline`** bundles preprocessing with the model, so the
preprocessing is re-fitted inside every training fold. This matters: if the
scaler learned the mean and spread from *all* patients before splitting,
information from the test patients would leak into training.

- Numeric predictors are **standardised** (rescaled to mean 0, spread 1).
  Logistic regression needs this so its penalty treats predictors equally;
  trees ignore it.
- Categories are **one-hot encoded** (one yes/no column per category).

In [ ]:
NUMERIC = ["age", "resting_bp", "cholesterol", "max_heart_rate", "st_depression"]
CATEGORICAL = ["sex", "chest_pain_type", "fasting_blood_sugar_gt_120", "resting_ecg",
               "exercise_angina", "st_slope"]
IMAGING_NUM, IMAGING_CAT = ["major_vessels"], ["thallium_result"]

def features(cols):
    X = cle[cols].copy()
    for c in X.columns:
        if c in NUMERIC + IMAGING_NUM:
            X[c] = X[c].astype(float)
        else:
            X[c] = X[c].astype(str)   # booleans and categories as plain labels
    return X

FEATURE_SETS = {
    "pre-imaging (11)": (NUMERIC, CATEGORICAL),
    "all 13": (NUMERIC + IMAGING_NUM, CATEGORICAL + IMAGING_CAT),
}

def make_models(num, cat):
    prep = ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])
    return {
        # C=1.0 is scikit-learn's default mild L2 penalty
        "Logistic regression": Pipeline([("prep", prep),
            ("model", LogisticRegression(max_iter=2000))]),
        "Random forest": Pipeline([("prep", prep),
            ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                             max_features="sqrt", random_state=0, n_jobs=-1))]),
        "Gradient boosting": Pipeline([("prep", prep),
            ("model", GradientBoostingClassifier(n_estimators=150, learning_rate=0.05,
                                                 max_depth=2, subsample=0.8, random_state=0))]),
    }
MODELS = list(make_models([], []).keys())
print("Hyperparameters fixed in advance:")
for name, pipe in make_models(NUMERIC, CATEGORICAL).items():
    print(f"  {name}: {pipe.named_steps['model']}")

**Why these settings?** They deliberately keep the trees simple:
`min_samples_leaf=5` stops the forest from memorising individual patients, and
boosting uses shallow trees (`max_depth=2`), a slow learning rate and random
subsampling. With under 300 patients, a more aggressive model would mostly
learn noise.

## 2. Cross-validated comparison

In [ ]:
N_REPEATS, N_FOLDS = 20, 10
rows, oof_store = [], {}
for fs_name, (num, cat) in FEATURE_SETS.items():
    X = features(num + cat)
    for rep in range(N_REPEATS):
        folds = StratifiedKFold(N_FOLDS, shuffle=True, random_state=rep)
        preds = {m: np.empty(len(y)) for m in MODELS}
        for train, test in folds.split(X, y):
            for name, pipe in make_models(num, cat).items():
                pipe.fit(X.iloc[train], y[train])
                preds[name][test] = pipe.predict_proba(X.iloc[test])[:, 1]
        for name, p in preds.items():
            rows.append({"features": fs_name, "model": name, "repeat": rep,
                         "AUC": roc_auc_score(y, p), "Brier": brier_score_loss(y, p)})
            if rep == 0:
                oof_store[(fs_name, name)] = p   # out-of-fold predictions for calibration
cv = pd.DataFrame(rows)

summary = (cv.groupby(["features", "model"], sort=False)
             .agg(AUC=("AUC", "mean"), AUC_low=("AUC", "min"), AUC_high=("AUC", "max"),
                  Brier=("Brier", "mean")).round(3))
summary.to_csv(ROOT / "reports" / "model_comparison.csv")
summary

`AUC_low` and `AUC_high` are the lowest and highest AUC across the 20
repeats: how much the score moves just from reshuffling the folds.

**Paired comparison.** Because every model saw identical splits, we can
compare them repeat by repeat. How often does each tree model beat logistic
regression, and by how much?

In [ ]:
wide = cv.pivot_table(index=["features", "repeat"], columns="model", values="AUC")
paired = []
for fs_name in FEATURE_SETS:
    w = wide.loc[fs_name]
    for m in ["Random forest", "Gradient boosting"]:
        d = w[m] - w["Logistic regression"]
        paired.append({"features": fs_name, "model": m,
                       "mean AUC difference vs logistic": round(d.mean(), 3),
                       "range": f"{d.min():+.3f} to {d.max():+.3f}",
                       "repeats where it wins": f"{(d > 0).sum()} of {len(d)}"})
pd.DataFrame(paired)

In [ ]:
SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
COLORS = {"Logistic regression": "#2a78d6", "Random forest": "#1baf7a", "Gradient boosting": "#4a3aa7"}
MARKERS = {"Logistic regression": "o", "Random forest": "s", "Gradient boosting": "D"}
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
                     "xtick.color": INK_2, "ytick.color": INK_2, "font.size": 9})

def tidy(ax):
    ax.tick_params(length=0)
    for s in ["top", "right", "left"]:
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(GRID)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True, gridspec_kw={"wspace": 0.16})
order = MODELS[::-1]
for ax, fs_name in zip(axes, FEATURE_SETS):
    s = summary.loc[fs_name].reindex(order)
    y_pos = np.arange(len(order))
    ax.hlines(y_pos, s["AUC_low"], s["AUC_high"], color=INK, linewidth=2, zorder=2)
    ax.scatter(s["AUC"], y_pos, s=60, color=INK, zorder=3)
    for yi, (m, r) in zip(y_pos, s.iterrows()):
        ax.text(r["AUC_high"] + 0.004, yi, f"{r['AUC']:.3f}", va="center", fontsize=8.5, color=INK_2)
    ax.set_yticks(y_pos, order)
    ax.set_xlim(0.82, 0.94)
    ax.set_ylim(-0.6, len(order) - 0.4)
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    tidy(ax)
    ax.set_title(f"Predictors: {fs_name}", loc="left", fontsize=10, color=INK)
    ax.set_xlabel("Cross-validated AUC", color=INK_2)
fig.suptitle("The simple model matched or beat the flexible ones", x=0.01, ha="left", fontsize=12, fontweight="bold", color=INK)
fig.text(0.01, 0.855, "Dot = mean AUC over 20 repeats of 10-fold cross-validation; line = lowest to highest repeat.",
         fontsize=8.5, color=INK_2)
fig.subplots_adjust(left=0.17, right=0.97, top=0.72, bottom=0.17)
fig.savefig(ROOT / "reports" / "figures" / "model_comparison_auc.png", dpi=200)
plt.show()

## 3. Calibration: are the predicted probabilities right?

A **calibration curve** groups patients by predicted risk (here into five
equal-sized groups) and plots the average predicted risk against the share
who actually had disease. A perfectly calibrated model sits on the diagonal:
of patients given a 70% risk, 70% have disease. Points below the diagonal mean
the model predicts more disease than there is; points above mean it predicts
less. (Out-of-fold predictions from the first repeat, pre-imaging
predictors. Brier scores in the legend are averages over all 20 repeats.)

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 5.2))
diag, = ax.plot([0, 1], [0, 1], color=INK_2, linewidth=1, linestyle=(0, (3, 3)), zorder=1,
                label="Perfect calibration")
handles = []
for name in MODELS:
    p = oof_store[("pre-imaging (11)", name)]
    groups = pd.qcut(p, 5, labels=False, duplicates="drop")
    cal = pd.DataFrame({"p": p, "y": y, "g": groups}).groupby("g").mean()
    h, = ax.plot(cal["p"], cal["y"], color=COLORS[name], marker=MARKERS[name], markersize=7,
                 linewidth=2, zorder=3, markeredgecolor=SURFACE, markeredgewidth=1,
                 label=f"{name} (Brier {summary.loc[('pre-imaging (11)', name), 'Brier']:.3f})")
    handles.append(h)
ax.legend(handles=handles + [diag], loc="upper left", frameon=False, fontsize=8.5)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel("Average predicted risk", color=INK_2)
ax.set_ylabel("Share who actually had disease", color=INK_2)
ax.grid(color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color(GRID); ax.spines["left"].set_color(GRID)
ax.tick_params(length=0)
ax.set_title("Calibration (pre-imaging predictors)", loc="left", fontsize=11,
             fontweight="bold", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "reports" / "figures" / "model_comparison_calibration.png", dpi=200)
plt.show()

## 4. Do the models rely on the same signals?

**Permutation importance** asks: if we randomly shuffle one predictor's values
(breaking its link with disease), how much does the model's AUC drop? A big
drop means the model leaned on that predictor. It works the same way for
every model type, so it lets us compare a logistic regression with a forest
on equal terms.

To avoid flattering the models, importance is measured on the **held-out**
fold each time (10 folds, 20 shuffles per predictor per fold), and whole
predictors are shuffled, not individual one-hot columns.

In [ ]:
num, cat = FEATURE_SETS["pre-imaging (11)"]
X = features(num + cat)
imp = {m: [] for m in MODELS}
for train, test in StratifiedKFold(10, shuffle=True, random_state=0).split(X, y):
    for name, pipe in make_models(num, cat).items():
        pipe.fit(X.iloc[train], y[train])
        r = permutation_importance(pipe, X.iloc[test], y[test], scoring="roc_auc",
                                   n_repeats=20, random_state=0)
        imp[name].append(r.importances_mean)
importance = pd.DataFrame({m: np.mean(v, axis=0) for m, v in imp.items()}, index=X.columns)
NICE = {"age": "Age", "resting_bp": "Resting BP", "cholesterol": "Cholesterol",
        "max_heart_rate": "Max heart rate", "st_depression": "ST depression", "sex": "Sex",
        "chest_pain_type": "Chest pain type", "fasting_blood_sugar_gt_120": "Fasting blood sugar",
        "resting_ecg": "Resting ECG", "exercise_angina": "Exercise angina", "st_slope": "ST slope"}
importance = importance.rename(index=NICE).sort_values("Logistic regression", ascending=False)
importance.round(3)

In [ ]:
print("Rank of each predictor by importance (1 = most important):")
importance.rank(ascending=False).astype(int)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 4.4), sharey=True, gridspec_kw={"wspace": 0.1})
rows_ = importance.index[::-1]
y_pos = np.arange(len(rows_))
for ax, name in zip(axes, MODELS):
    vals = importance.loc[rows_, name]
    ax.barh(y_pos, vals, height=0.62, color=COLORS[name])
    ax.axvline(0, color=INK_2, linewidth=0.8)
    ax.set_yticks(y_pos, rows_)
    ax.set_xlim(min(-0.01, importance.values.min() - 0.005), importance.values.max() * 1.1)
    ax.grid(axis="x", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
    tidy(ax)
    ax.set_title(name, loc="left", fontsize=10, color=INK)
    ax.set_xlabel("Drop in AUC when shuffled", color=INK_2, fontsize=8.5)
fig.suptitle("All three models agree on the top three signals", x=0.01, ha="left", fontsize=12, fontweight="bold", color=INK)
fig.text(0.01, 0.9, "Permutation importance on held-out folds, pre-imaging predictors. "
         "Rows sorted by the logistic regression's ranking.", fontsize=8.5, color=INK_2)
fig.subplots_adjust(left=0.14, right=0.98, top=0.8, bottom=0.13)
fig.savefig(ROOT / "reports" / "figures" / "model_comparison_importance.png", dpi=200)
plt.show()

## 5. What the comparison shows

**The simple model holds its own.** With the 11 routine predictors, logistic
regression scored the highest cross-validated AUC (0.872), ahead of the
random forest (0.865) and gradient boosting (0.855). Neither tree model beat
it in a single one of the 20 repeats. With all 13 predictors, the random
forest essentially tied it (0.909 vs 0.908, ahead in 13 of 20 repeats by at
most 0.006), and boosting again trailed. Logistic regression also had the
best Brier score in both cases, so its probabilities were the most accurate
as well as its ranking.

**Why the flexible models didn't win here:**
- **Small data.** 297 patients is not much for a model that has to discover
  interactions and thresholds on its own; a model with fewer moving parts
  wastes less of the data learning noise.
- **The signal is mostly additive.** The predictors that matter (chest pain
  type, sex, ST depression) seem to shift risk in a fairly steady way, which
  is exactly what logistic regression assumes.
- **Fixed, conservative settings.** The tree models weren't tuned. Nested
  cross-validation might close part of the gap, but with a gap this small and
  consistent, it's unlikely to reverse it.

**They agree on what matters.** All three models rank the same three
predictors on top, in the same order: chest pain type, sex and ST depression.
Cholesterol, resting ECG and fasting blood sugar sit near zero for all three,
matching the conclusions of notebooks 04 to 06. That agreement between very
different kinds of model is itself evidence the signals are real.

One difference is worth noting: the tree models gave **age** more weight
(ranked 5th and 6th) than logistic regression did (10th). Trees can use age
in combination with other predictors, which a model without interaction terms
can't. It's a lead for follow-up rather than a finding: the effect is small
and the trees didn't predict better overall.

**The takeaway:** a more complex model is only worth its cost in
interpretability if it predicts measurably better. Here it doesn't, so the
logistic regression from notebook 05, whose odds ratios can be explained to
a clinician, is the right model to keep.

**Next steps:** nested cross-validation to tune the tree models fairly, and
checking the comparison on the four-site data from notebook 06.